# 기본 정보 입력

In [9]:
# 패키지 불러오기
import openai
import yfinance as yf
import json
import os
from dotenv import load_dotenv

# Custom Functions 생성

In [10]:
def get_stock_price(symbol):
    try:
        stock = yf.Ticker(symbol)
        price = stock.info.get('currentPrice', 'N/A')
        return price if price else 'N/A'
    except Exception as e:
        return f"주가 조회 실패: {str(e)}"

In [11]:
def get_latest_company_news(symbol):
    try:
        stock = yf.Ticker(symbol)
        news = stock.news
        # 최신 뉴스 3개 리스트에 저장하기
        news_list = []
        num = 1
        for item in news[:3]:
            try:
                content = item.get('content', {})
                provider = content.get('provider', {})
                click_url = content.get('clickThroughUrl') or {}
                
                title = content.get('title', 'N/A')
                publisher = provider.get('displayName', 'N/A')
                link = click_url.get('url', 'N/A') if click_url else 'N/A'
                
                news_list.append(f"{num}: title : {title}, publisher : {publisher}, link : {link}")
                num += 1
            except Exception as e:
                continue
        return news_list
    except Exception as e:
        return [f"뉴스 조회 실패: {str(e)}"]

In [12]:
tools_list = [{"type":"function","name":"get_stock_price","description":"현재 주가를 조회합니다.","parameters":{"type":"object","properties":{"symbol":{"type":"string"}},"required":["symbol"],"additionalProperties":False}},{"type":"function","name":"get_latest_company_news","description":"최신 기업 뉴스를 조회합니다.","parameters":{"type":"object","properties":{"symbol":{"type":"string"}},"required":["symbol"],"additionalProperties":False}}]


In [13]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [14]:
# API 키 지정하여 클라이언트 선언하기
client = openai.OpenAI(api_key = api_key)

In [15]:
question="TSLA의 현재 주가와 최신 뉴스를 알려줘."
response=client.responses.create(
    model="gpt-4o-mini",
    instructions="항상 한국어로 답변하세요. 주가와 뉴스는 도구 결과만 사용하세요.",
    input=question,tools=tools_list)
for _ in range(3):
    calls=[x for x in response.output if x.type=="function_call"]
    if not calls: break
    outputs=[]
    for call in calls:
        args=json.loads(call.arguments)
        value=get_stock_price(args["symbol"]) if call.name=="get_stock_price" else get_latest_company_news(args["symbol"])
        outputs.append({
            "type":"function_call_output",""
            "call_id":call.call_id,
            "output":json.dumps(value,ensure_ascii=False)})

    response=client.responses.create(
        model="gpt-4o-mini",
        instructions="항상 한국어로 답변하세요.",
        input=[*response.output,*outputs],
        tools=tools_list)
    
print(response.output_text)


### 현재 주가
- **테슬라 (TSLA)**: $363.56

### 최신 기업 뉴스
1. **[TSLA Stock Rises Premarket On Global Blitz](https://finance.yahoo.com/markets/stocks/articles/tsla-stock-rises-premarket-global-080920657.html)**  
   - 출처: Stocktwits
   - 내용: 일본에서의 사이버택시, 중국에서의 개선된 모델 Y, 미국의 파워셰어로 인해 테슬라 주가가 상승.

2. **[BYD Outsold Tesla by Roughly 77,000 EVs Again in Q2. Can Tesla Close the Gap?](https://finance.yahoo.com/markets/stocks/articles/byd-outsold-tesla-roughly-77-075000803.html)**  
   - 출처: Motley Fool
   - 내용: BYD가 2분기 동안 테슬라를 약 77,000대의 전기차로 초과 판매. 테슬라가 격차를 좁힐 수 있을까에 대한 논의.

3. **[Tesla (TSLA) Falls More Steeply Than Broader Market: What Investors Need to Know](https://finance.yahoo.com/markets/stocks/articles/tesla-tsla-falls-more-steeply-214505362.html)**  
   - 출처: Zacks
   - 내용: 테슬라 주가가 전반적인 시장보다 더 크게 하락하고 있다는 정보.

더 궁금한 사항이 있으신가요?


## Function Calling 실습 단계별 확인

In [16]:
# 1단계: 사용자의 질문을 모델에 전달합니다.
question = "TSLA의 현재 주가와 최신 뉴스를 알려줘."
response = client.responses.create(
    model="gpt-4o-mini",
    instructions="항상 한국어로 답변하세요. 주가와 뉴스는 도구 결과만 사용하세요.",
    input=question,
    tools=tools_list,
)


In [ ]:
# 2단계: 모델이 함수를 호출했는지 확인합니다.
calls = [item for item in response.output if item.type == "function_call"]
for call in calls:
    print("함수명:", call.name)
    print("인자:", call.arguments)


In [ ]:
# 3단계: 모델이 요청한 함수를 실제 Python 함수로 실행합니다.
tool_outputs = []
for call in calls:
    arguments = json.loads(call.arguments)
    if call.name == "get_stock_price":
        result = get_stock_price(arguments["symbol"])
    elif call.name == "get_latest_company_news":
        result = get_latest_company_news(arguments["symbol"])
    tool_outputs.append({
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, ensure_ascii=False),
    })
print(tool_outputs)


In [ ]:
# 4단계: 함수 결과를 모델에 전달하고 최종 답변을 받습니다.
final_response = client.responses.create(
    model="gpt-4o-mini",
    instructions="반드시 한국어로 답변하세요.",
    input=[*response.output, *tool_outputs],
    tools=tools_list,
)
print(final_response.output_text)


## 함수 호출이 필요 없는 질문

In [ ]:
# 일반 질문에는 함수 호출 없이 모델이 바로 답변할 수 있습니다.
general_response = client.responses.create(
    model="gpt-4o-mini",
    instructions="반드시 한국어로 답변하세요.",
    input="Function Calling이 무엇인지 설명해 주세요.",
    tools=tools_list,
)
print(general_response.output_text)
